# Notebook to test GPU training on cloud computing

This notebook expects only a single CUDA or MPS GPU to be available.

In [ ]:
from time import time

import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"torch.accelerater.current_accelerator() gives {device} device")
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#print(f"CUDA device is {device}")

# First benchmark: matrix multiplication

In [ ]:
B, N = 64, 1024
num_repeat = 5

In [ ]:
# First: CPU
M1 = torch.randn(B, N, N).to(device='cpu')
M2 = torch.randn(B, N, N).to(device='cpu')

# JIT compilation happens
res = torch.bmm(M1, M2)

t1 = time()
for _ in range(num_repeat):
    res = torch.bmm(M1, M2)
t2 = time()
print(f"Time per matrix multiplication on cpu is {(t2-t1)*1000/num_repeat:.2f} ms")

In [ ]:
# Second: MPS or CUDA device
if device == 'mps':
    M1 = torch.randn(B, N, N).to(device)
    M2 = torch.randn(B, N, N).to(device)

    # JIT compilation happens
    res = torch.bmm(M1, M2)

    t1 = time()
    for _ in range(num_repeat):
        res = torch.bmm(M1, M2)
    if device == 'mps':
        torch.mps.synchronize()
    elif device == 'cuda':
        torch.cuda.synchronize()
    else:
        raise ValueError(f"Device {device} not expected")
    t2 = time()
    print(f"Time per matrix multiplication on {device} is {(t2-t1)*1000/num_repeat:.2f} ms")

# Second benchmark: train on MNIST 

In [ ]:
batch_size = 64

In [ ]:
train_ds = MNIST('data', train=True, download=True, transform=transforms.ToTensor())
val_ds = MNIST('data', train=False, download=True, transform=transforms.ToTensor())
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size)

In [ ]:
class MNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layers = nn.ModuleList()
        self.linear_layers.append(nn.Linear(784, 64))
        self.linear_layers.append(nn.Linear(64, 64))
        self.linear_layers.append(nn.Linear(64, 64))
        self.linear_layers.append(nn.Linear(64, 64))
        self.linear_out = nn.Linear(64, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        for layer in self.linear_layers:
            x = self.relu(layer(x))
        return self.linear_out(x)

In [ ]:
num_epochs = 5
lr = 0.05

In [ ]:
def step(model, opt, X, y):
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    opt.step()
    opt.zero_grad()
    return loss.item()

def val_stats(model, X, y):
    with torch.no_grad():
        logits = model(X)
        loss = F.cross_entropy(logits, y)
    return loss.item()

In [ ]:
# First: timing for CPU

model = MNISTMLP().to('cpu')
opt = torch.optim.SGD(model.parameters(), lr)

t1 = time()

for epoch in range(num_epochs):
    for X, y in train_dl:
        X = X.to('cpu')
        y = y.to('cpu')
        _ = step(model, opt, X, y)

    for X, y in val_dl:
        X = X.to('cpu')
        y = y.to('cpu')
        _ = val_stats(model, X, y)

t2 = time()

print(f"Time to train MNIST on cpu is {(t2-t1):.2f} seconds")

In [ ]:
# Second: timing for mps and cuda

model = MNISTMLP().to(device)
opt = torch.optim.SGD(model.parameters(), lr)

t1 = time()

for epoch in range(num_epochs):
    for X, y in train_dl:
        X = X.to(device)
        y = y.to(device)
        _ = step(model, opt, X, y)

    for X, y in val_dl:
        X = X.to(device)
        y = y.to(device)
        _ = val_stats(model, X, y)

if device == 'mps':
    torch.mps.synchronize()
elif device == 'cuda':
    torch.cuda.synchronize()
else:
    raise ValueError(f"Device {device} not expected")

t2 = time()

print(f"Time to train MNIST on {device} is {(t2-t1):.2f} seconds")